In [1]:
import os 
import yaml

In [2]:
# read config file

config_file_path = './data_provider/multi_task_pretrain.yaml'
# print(config_file_path)

with open(config_file_path, 'r') as config_file:
    config = yaml.load(config_file, Loader=yaml.FullLoader)
task_dataset_config = config.get('task_dataset', {})

print(f'type of dataset config: {type(task_dataset_config)}')
# print(task_dataset_config)
# task_dataset_config['NN5_p112']
task_dataset_config

type of dataset config: <class 'dict'>


{'NN5_p112': {'task_name': 'pretrain_long_term_forecast',
  'dataset_name': 'nn5_daily_without_missing',
  'dataset': 'NN5',
  'data': 'gluonts',
  'root_path': '../dataset/gluonts',
  'seq_len': 224,
  'label_len': 0,
  'pred_len': 0,
  'features': 'M',
  'embed': 'timeF',
  'enc_in': 111,
  'dec_in': 111,
  'c_out': 111},
 'LTF_ECL_p96': {'task_name': 'pretrain_long_term_forecast',
  'dataset': 'ECL',
  'data': 'custom',
  'embed': 'timeF',
  'root_path': '../dataset/electricity/',
  'data_path': 'electricity.csv',
  'features': 'M',
  'seq_len': 192,
  'label_len': 48,
  'pred_len': 0,
  'enc_in': 321,
  'dec_in': 321,
  'c_out': 321},
 'LTF_ECL_p192': {'task_name': 'pretrain_long_term_forecast',
  'dataset': 'ECL',
  'data': 'custom',
  'embed': 'timeF',
  'root_path': '../dataset/electricity/',
  'data_path': 'electricity.csv',
  'features': 'M',
  'seq_len': 288,
  'label_len': 48,
  'pred_len': 0,
  'enc_in': 321,
  'dec_in': 321,
  'c_out': 321},
 'LTF_ECL_p336': {'task_name': 

In [3]:
task_data_config_list = []
default_batch_size = 64

for task_name, task_config in task_dataset_config.items():
    # print(f'Task Name: {task_name}')
    # print(f'Task Config: {task_config}')
    task_config['max_batch'] = default_batch_size
    task_data_config_list.append([task_name, task_config])
    # break 

task_data_config_list

[['NN5_p112',
  {'task_name': 'pretrain_long_term_forecast',
   'dataset_name': 'nn5_daily_without_missing',
   'dataset': 'NN5',
   'data': 'gluonts',
   'root_path': '../dataset/gluonts',
   'seq_len': 224,
   'label_len': 0,
   'pred_len': 0,
   'features': 'M',
   'embed': 'timeF',
   'enc_in': 111,
   'dec_in': 111,
   'c_out': 111,
   'max_batch': 64}],
 ['LTF_ECL_p96',
  {'task_name': 'pretrain_long_term_forecast',
   'dataset': 'ECL',
   'data': 'custom',
   'embed': 'timeF',
   'root_path': '../dataset/electricity/',
   'data_path': 'electricity.csv',
   'features': 'M',
   'seq_len': 192,
   'label_len': 48,
   'pred_len': 0,
   'enc_in': 321,
   'dec_in': 321,
   'c_out': 321,
   'max_batch': 64}],
 ['LTF_ECL_p192',
  {'task_name': 'pretrain_long_term_forecast',
   'dataset': 'ECL',
   'data': 'custom',
   'embed': 'timeF',
   'root_path': '../dataset/electricity/',
   'data_path': 'electricity.csv',
   'features': 'M',
   'seq_len': 288,
   'label_len': 48,
   'pred_len': 0

# read data

In [4]:
import pandas as pd 

# fixed data_path 
ETT_smal_data_path = './dataset/electricity'

# data_name = 'ETTm1.csv'
dataset_name = 'electricity.csv'
data_path = os.path.join(ETT_smal_data_path, dataset_name)

df = pd.read_csv(data_path)
df.head()

,date,0,1,2,3,4,5,6,7,8,...,311,312,313,314,315,316,317,318,319,OT
0,2016-07-01 02:00:00,14.0,69.0,234.0,415.0,215.0,1056.0,29.0,840.0,226.0,...,676.0,372.0,80100.0,4719.0,5002.0,48.0,38.0,1558.0,182.0,2162.0
1,2016-07-01 03:00:00,18.0,92.0,312.0,556.0,292.0,1363.0,29.0,1102.0,271.0,...,805.0,452.0,95200.0,4643.0,6617.0,65.0,47.0,2177.0,253.0,2835.0
2,2016-07-01 04:00:00,21.0,96.0,312.0,560.0,272.0,1240.0,29.0,1025.0,270.0,...,817.0,430.0,96600.0,4285.0,6571.0,64.0,43.0,2193.0,218.0,2764.0
3,2016-07-01 05:00:00,20.0,92.0,312.0,443.0,213.0,845.0,24.0,833.0,179.0,...,801.0,291.0,94500.0,4222.0,6365.0,65.0,39.0,1315.0,195.0,2735.0
4,2016-07-01 06:00:00,22.0,91.0,312.0,346.0,190.0,647.0,16.0,733.0,186.0,...,807.0,279.0,91300.0,4116.0,6298.0,75.0,40.0,1378.0,191.0,2721.0


In [60]:
from sklearn.preprocessing import StandardScaler
# standardScaler: make the data have zero mean and unit variance

scaler = StandardScaler()

# make train/test/valid index
flag = 'train'
type_map = {'train': 0, 'val': 1, 'test': 2}
set_type = type_map[flag]
print(f'set type: {set_type}')

# hyperprams
print(f'dataset: {dataset_name}')
# LTF_ECL_p96 = electricity 
config_dataset = task_dataset_config['LTF_ECL_p96']
seq_len = config_dataset['seq_len']
features = config_dataset['features']
scale = True
timeenc = 1

df.head()

set type: 0
dataset: electricity.csv


,date,0,1,2,3,4,5,6,7,8,...,311,312,313,314,315,316,317,318,319,OT
0,2016-07-01 02:00:00,14.0,69.0,234.0,415.0,215.0,1056.0,29.0,840.0,226.0,...,676.0,372.0,80100.0,4719.0,5002.0,48.0,38.0,1558.0,182.0,2162.0
1,2016-07-01 03:00:00,18.0,92.0,312.0,556.0,292.0,1363.0,29.0,1102.0,271.0,...,805.0,452.0,95200.0,4643.0,6617.0,65.0,47.0,2177.0,253.0,2835.0
2,2016-07-01 04:00:00,21.0,96.0,312.0,560.0,272.0,1240.0,29.0,1025.0,270.0,...,817.0,430.0,96600.0,4285.0,6571.0,64.0,43.0,2193.0,218.0,2764.0
3,2016-07-01 05:00:00,20.0,92.0,312.0,443.0,213.0,845.0,24.0,833.0,179.0,...,801.0,291.0,94500.0,4222.0,6365.0,65.0,39.0,1315.0,195.0,2735.0
4,2016-07-01 06:00:00,22.0,91.0,312.0,346.0,190.0,647.0,16.0,733.0,186.0,...,807.0,279.0,91300.0,4116.0,6298.0,75.0,40.0,1378.0,191.0,2721.0


In [19]:
def slash(n=30):
    print('-' * n)

slash()

------------------------------


In [62]:
from typing import List

import numpy as np
import pandas as pd
from pandas.tseries import offsets
from pandas.tseries.frequencies import to_offset


class TimeFeature:
    def __init__(self):
        pass

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        pass

    def __repr__(self):
        return self.__class__.__name__ + "()"


class SecondOfMinute(TimeFeature):
    """Minute of hour encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return index.second / 59.0 - 0.5


class MinuteOfHour(TimeFeature):
    """Minute of hour encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return index.minute / 59.0 - 0.5


class HourOfDay(TimeFeature):
    """Hour of day encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return index.hour / 23.0 - 0.5


class DayOfWeek(TimeFeature):
    """Hour of day encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return index.dayofweek / 6.0 - 0.5


class DayOfMonth(TimeFeature):
    """Day of month encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return (index.day - 1) / 30.0 - 0.5


class DayOfYear(TimeFeature):
    """Day of year encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return (index.dayofyear - 1) / 365.0 - 0.5


class MonthOfYear(TimeFeature):
    """Month of year encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return (index.month - 1) / 11.0 - 0.5


class WeekOfYear(TimeFeature):
    """Week of year encoded as value between [-0.5, 0.5]"""

    def __call__(self, index: pd.DatetimeIndex) -> np.ndarray:
        return (index.isocalendar().week - 1) / 52.0 - 0.5


def time_features_from_frequency_str(freq_str: str) -> List[TimeFeature]:
    """
    Returns a list of time features that will be appropriate for the given frequency string.
    Parameters
    ----------
    freq_str
        Frequency string of the form [multiple][granularity] such as "12H", "5min", "1D" etc.
    """

    features_by_offsets = {
        offsets.YearEnd: [],
        offsets.QuarterEnd: [MonthOfYear],
        offsets.MonthEnd: [MonthOfYear],
        offsets.Week: [DayOfMonth, WeekOfYear],
        offsets.Day: [DayOfWeek, DayOfMonth, DayOfYear],
        offsets.BusinessDay: [DayOfWeek, DayOfMonth, DayOfYear],
        offsets.Hour: [HourOfDay, DayOfWeek, DayOfMonth, DayOfYear],
        offsets.Minute: [
            MinuteOfHour,
            HourOfDay,
            DayOfWeek,
            DayOfMonth,
            DayOfYear,
        ],
        offsets.Second: [
            SecondOfMinute,
            MinuteOfHour,
            HourOfDay,
            DayOfWeek,
            DayOfMonth,
            DayOfYear,
        ],
    }

    offset = to_offset(freq_str)

    for offset_type, feature_classes in features_by_offsets.items():
        if isinstance(offset, offset_type):
            return [cls() for cls in feature_classes]

    supported_freq_msg = f"""
    Unsupported frequency {freq_str}
    The following frequencies are supported:
        Y   - yearly
            alias: A
        M   - monthly
        W   - weekly
        D   - daily
        B   - business days
        H   - hourly
        T   - minutely
            alias: min
        S   - secondly
    """
    raise RuntimeError(supported_freq_msg)


def time_features(dates, freq='h'):
    return np.vstack([feat(dates) for feat in time_features_from_frequency_str(freq)])

In [64]:
to_offset('h')

<Hour>

In [72]:
cols = list(df.columns)
print(f'len cols in df: {len(cols)}')
print(cols)

cols.remove('date')
cols.remove('OT')

# re-order 
df = df[['date'] + cols + ['OT']]
# df.head()

num_train = int(len(df) * 0.7)
num_test = int(len(df) * 0.2)
num_val = len(df) - num_train - num_test
print(f'num_train: {num_train}, num_test: {num_test}, num_val: {num_val}') 

slash()
border1s = [0, num_train - seq_len,len(df) - num_test - seq_len]
border2s = [num_train, num_train + num_val, len(df)]
print(f'border1s: {border1s}')
print(f'border2s: {border2s}')
border1 = border1s[set_type]
border2 = border2s[set_type]

if features == 'M':
    # features = M: all features 
    cols_data = df.columns[1:]
    # print(cols_data)
    df_data = df[cols_data]
# df_data.head()

if scale:
    train_data = df_data[border1s[0]:border2s[0]]
    # print(train_data.head())
    # print(train_data.values.shape)
    scaler.fit(train_data.values)
    data = scaler.transform(df_data.values)
else:
    data = df_data.values

df_stamp = df[['date']][border1:border2]
print(type(df_stamp.date))
df_stamp['date'] = pd.to_datetime(df_stamp.date)
df_stamp.head()
print(type(df_stamp.date))

if timeenc == 0:
    # timeenc = 0: use month, day, weekday, hour as features
    df_stamp['month'] = df_stamp.date.apply(lambda row: row.month)
    df_stamp['day'] = df_stamp.date.apply(lambda row: row.day)
    df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday())
    df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour)
    data_stamp = df_stamp.drop(columns=['date']).values
    # df_stamp.head()
    # data_stamp
elif timeenc == 1:
    data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq='h')
    # print(data_stamp.shape)
    # print(data_stamp)
    data_stamp = data_stamp.transpose(1, 0) # [seq_len, num_features]

data_x = data[border1:border2]
data_y = data[border1:border2]
data_stamp = data_stamp 
print(data_x.shape)

len cols in df: 322
['date', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '15

In [92]:
# dataset modeling
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, data_path, seq_len, set_type=0, pred_len=48, label_len=24):
        self.data_path = data_path
        self.seq_len = seq_len
        self.set_type = set_type
        self.pred_len = pred_len
        self.label_len = label_len

        self.__read_data()

    def __read_data(self):
        self.scaler = StandardScaler()

        # read_data from .csv
        df = pd.read_csv(self.data_path)

        cols = list(df.columns)
        # print(f'len cols in df: {len(cols)}')
        # print(cols)

        cols.remove('date')
        cols.remove('OT')

        # re-order 
        df = df[['date'] + cols + ['OT']]
        # df.head()

        num_train = int(len(df) * 0.7)
        num_test = int(len(df) * 0.2)
        num_val = len(df) - num_train - num_test
        print(f'num_train: {num_train}, num_test: {num_test}, num_val: {num_val}') 

        slash()
        border1s = [0, num_train - self.seq_len, len(df) - num_test - self.seq_len]
        border2s = [num_train, num_train + num_val, len(df)]
        # print(f'border1s: {border1s}')
        # print(f'border2s: {border2s}')
        border1 = border1s[self.set_type]
        border2 = border2s[self.set_type]

        if features == 'M':
            # features = M: all features 
            cols_data = df.columns[1:]
            # print(cols_data)
            df_data = df[cols_data]
        # df_data.head()

        if scale:
            train_data = df_data[border1s[0]:border2s[0]]
            # print(train_data.head())
            # print(train_data.values.shape)
            scaler.fit(train_data.values)
            data = scaler.transform(df_data.values)
        else:
            data = df_data.values

        df_stamp = df[['date']][border1:border2]
        print(type(df_stamp.date))
        df_stamp['date'] = pd.to_datetime(df_stamp.date)
        df_stamp.head()
        print(type(df_stamp.date))

        if timeenc == 0:
            # timeenc = 0: use month, day, weekday, hour as features
            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month)
            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day)
            df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday())
            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour)
            data_stamp = df_stamp.drop(columns=['date']).values
            # df_stamp.head()
            # data_stamp
        elif timeenc == 1:
            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq='h')
            # print(data_stamp.shape)
            # print(data_stamp)
            data_stamp = data_stamp.transpose(1, 0) # [seq_len, num_features]

        self.data_x = data[border1:border2]
        self.data_y = data[border1:border2]
        self.data_stamp = data_stamp 
        print(data_x.shape)

    def __len__(self):
        return len(self.data_x) - self.seq_len - self.pred_len + 1 
        
    def __getitem__(self, idx):
        s_begin = idx
        s_end = s_begin + self.seq_len
        print(f's_begin: {s_begin}, s_end: {s_end}')

        """
        phần label_len đầu của seq_y bị chồng lên với đuôi của seq)x có tác dụng:
        làm context cho decoder đự đoán 
        """

        r_begin = s_end - self.label_len
        r_end = r_begin + self.label_len + self.pred_len

        seq_x = self.data_x[s_begin:s_end]
        seq_y = self.data_y[r_begin:r_end]
        seq_x_mark = self.data_stamp[s_begin:s_end]
        seq_y_mark = self.data_stamp[r_begin:r_end]

        return seq_x, seq_y, seq_x_mark, seq_y_mark



data_path = os.path.join("dataset\electricity\electricity.csv")
train_dataset = CustomDataset(data_path, seq_len, pred_len=48, label_len=24)
print(f'len train dataset: {len(train_dataset)}')

# debug for __getitem__
slash()
print('debug for __getitem__')
idx = 0
seq_x, seq_y, seq_x_mark, seq_y_mark = train_dataset[idx]
print(f'seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}')

slash()

# print(seq_len) # 192

num_train: 18412, num_test: 5260, num_val: 2632
------------------------------
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
(18412, 321)
len train dataset: 18173
------------------------------
debug for __getitem__
s_begin: 0, s_end: 192
seq_x shape: (192, 321), seq_y shape: (72, 321)
------------------------------


In [6]:
# print(list(task_dataset_config.keys()))
task_dataset_config['LTF_ECL_p96']

{'task_name': 'pretrain_long_term_forecast',
 'dataset': 'ECL',
 'data': 'custom',
 'embed': 'timeF',
 'root_path': '../dataset/electricity/',
 'data_path': 'electricity.csv',
 'features': 'M',
 'seq_len': 192,
 'label_len': 48,
 'pred_len': 0,
 'enc_in': 321,
 'dec_in': 321,
 'c_out': 321,
 'max_batch': 64}